# Étape 2 : Prétraitement des images
Objectif : transformer les images en données exploitables, puis séparer les données en entraînement (80 %) et test (20 %) **sans fuite**, en gardant toutes les photos d'une même fève dans la même partition (groupes identifiés à l'étape 1).

In [ ]:
#bibliothèques et inventaire de l’étape 1
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

DATA_DIR = "../data/feve"
GRAINE = 42   # pour que les découpages soient reproductibles

# Inventaire construit à l'étape 1 (avec la colonne "groupe")
df = pd.read_csv("../results/tables/01_inventaire_images.csv")

# Les classes deviennent des numéros : category 1 -> 0, category 2 -> 1, category 3 -> 2
classes = sorted(df["classe"].unique())
df["label"] = df["classe"].map({c: i for i, c in enumerate(classes)})
print(classes)
df.head()

['category 1', 'category 2', 'category 3']


,classe,fichier,largeur,hauteur,mode,md5,groupe,label
0,category 1,feve_731.jpg,60,60,RGB,4d0141f347ffce4eae4e4351c1fb2723,0,0
1,category 1,feve_725.jpg,60,60,RGB,7460fe30a4d384dfc2dcc5c40525ace2,1,0
2,category 1,feve_1019.jpg,60,60,RGB,d71fe390bcdc8776c6ab861030d60f50,2,0
3,category 1,feve_1031.jpg,60,60,RGB,554fb7371acfbf25aef28e5e413bc0a9,3,0
4,category 1,feve_1757.jpg,60,60,RGB,267c01caedbade777c593b185f8a4f13,4,0


In [2]:
# Lecture de toutes les images dans un seul tableau numpy
X = np.array([np.asarray(Image.open(os.path.join(DATA_DIR, c, f)).convert("RGB"))
              for c, f in zip(df["classe"], df["fichier"])])
y = df["label"].values
groupes = df["groupe"].values

print("Forme de X :", X.shape, "| type :", X.dtype)
print("Valeurs min / max :", X.min(), "/", X.max())

# Normalisation : pixels entre 0 et 1
X_norm = X.astype("float32") / 255.0
print("Après normalisation :", X_norm.min(), "/", X_norm.max())

Forme de X : (4470, 60, 60, 3) | type : uint8
Valeurs min / max : 0 / 255
Après normalisation : 0.0 / 1.0


In [3]:
#Sséparation entraînement / test sans fuite

# 5 morceaux stratifiés par groupe -> on garde 1 morceau pour le test (20 %)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=GRAINE)
idx_train, idx_test = next(sgkf.split(X_norm, y, groups=groupes))

print("Entraînement :", len(idx_train), "images | Test :", len(idx_test), "images")

# Vérification 1 : aucune fève à la fois dans train et test
fuite = set(groupes[idx_train]) & set(groupes[idx_test])
print("Groupes communs entre train et test :", len(fuite))

# Vérification 2 : les proportions de classes sont conservées
prop = pd.DataFrame({
    "Total (%)": pd.Series(y).value_counts(normalize=True).sort_index() * 100,
    "Train (%)": pd.Series(y[idx_train]).value_counts(normalize=True).sort_index() * 100,
    "Test (%)": pd.Series(y[idx_test]).value_counts(normalize=True).sort_index() * 100,
}).round(1)
prop.index = classes
prop

Entraînement : 3576 images | Test : 894 images
Groupes communs entre train et test : 0


,Total (%),Train (%),Test (%)
category 1,42.9,42.9,42.8
category 2,37.5,37.5,37.5
category 3,19.6,19.6,19.7


**Résultat de la séparation :** 3 576 images en entraînement (80 %) et 894 en test (20 %). Aucun groupe de fèves n'est présent dans les deux ensembles, ce qui empêche la fuite de données. Grâce à la stratification, la répartition des classes est conservée à 0,1 % près dans chaque ensemble.